
# Практическое занятие: word2vec + классификация текстов (ИНН)


In [7]:
# ---------------------------
# 0. Настройки и зависимости
# ---------------------------

import os
import re
import numpy as np
import pandas as pd

import nltk
from nltk.corpus import stopwords

import gensim
from gensim.models import Word2Vec

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Для русского языка можно поставить pymorphy2, если нужно:
# pip install pymorphy2
# import pymorphy2

# ---------------------------
# 1. Параметры
# ---------------------------

# Путь к CSV с данными:
# в формате: text, label

DATA_PATH = "data.csv"  

# Параметры word2vec
W2V_VECTOR_SIZE = 200
W2V_WINDOW = 7
W2V_MIN_COUNT = 3
W2V_EPOCHS = 20
W2V_SG = 1  # 1 = Skip-Gram, 0 = CBOW

# Параметры нейросети
HIDDEN_DIM = 128
DROPOUT = 0.3
BATCH_SIZE = 64
EPOCHS_NN = 10
LEARNING_RATE = 1e-3

# Доля тестовой выборки
TEST_SIZE = 0.3
RANDOM_STATE = 7

# Язык стоп-слов nltk: 
LANG = "russian"

In [12]:
# ---------------------------
# 2. Загрузка данных
# ---------------------------

print("Загрузка данных...")

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Файл {DATA_PATH} не найден."
    )

df = pd.read_csv(DATA_PATH, sep=";",encoding="utf-8")

# Проверка колонок
if "text" not in df.columns or "label" not in df.columns:
    # Попытка угадать: ищем первую текстовую и первую целевую колонку
    text_col = [c for c in df.columns if df[c].dtype == "object" and c.lower() in ["text", "review", "content", "body"]][:1]
    label_col = [c for c in df.columns if c.lower() in ["label", "class", "target", "y"]][:1]
    if not (text_col and label_col):
        raise ValueError(
            "В CSV не найдены колонки 'text' и 'label'. "
            "Переименуйте их или поправьте код (text_col, label_col)."
        )
    text_col = text_col[0]
    label_col = label_col[0]
    print(f"Используем колонки: text='{text_col}', label='{label_col}'")
    texts = df[text_col].astype(str).tolist()
    labels = df[label_col].tolist()
else:
    texts = df["text"].astype(str).tolist()
    labels = df["label"].tolist()

print(f"Всего документов: {len(texts)}, классов: {len(set(labels))}")

Загрузка данных...
Всего документов: 44900, классов: 2


In [13]:
# ---------------------------
# 3. Предобработка текста
# ---------------------------

print("Загрузка стоп-слов и предобработка...")

try:
    stop_words = set(stopwords.words(LANG))
except LookupError:
    nltk.download("stopwords")
    stop_words = set(stopwords.words(LANG))


def preprocess_text(text: str):
    # Приведение к нижнему регистру
    text = text.lower()
    # Оставляем буквы (кириллица + латиница) и пробелы
    text = re.sub(r"[^а-яёa-z\s]", " ", text)
    # Токенизация по пробелам
    tokens = text.split()
    # Удаление стоп-слов и слишком коротких токенов
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    return tokens

docs_tokens = [preprocess_text(t) for t in texts]

# Убираем пустые документы (если вдруг все токены выкинулись)
non_empty = [i for i, t in enumerate(docs_tokens) if len(t) > 0]
if len(non_empty) < len(docs_tokens):
    print(f"Удалено пустых документов после предобработки: {len(docs_tokens) - len(non_empty)}")
docs_tokens = [docs_tokens[i] for i in non_empty]
labels = [labels[i] for i in non_empty]

print(f"Документов после предобработки: {len(docs_tokens)}")


Загрузка стоп-слов и предобработка...
Удалено пустых документов после предобработки: 21
Документов после предобработки: 44879


In [16]:
# ---------------------------
# 4. Обучение word2vec
# ---------------------------

print("Обучение word2vec...")

model_w2v = Word2Vec(
    sentences=docs_tokens,
    size=W2V_VECTOR_SIZE, #vector_
    window=W2V_WINDOW,
    min_count=W2V_MIN_COUNT,
    workers=4,
    sg=W2V_SG,
    negative=5,
    iter=W2V_EPOCHS, #epochs
)



Обучение word2vec...


TypeError: object of type 'Word2VecKeyedVectors' has no len()

In [20]:
print(f"Размер словаря word2vec: {len(model_w2v.wv.vocab)}")



Размер словаря word2vec: 36875


In [23]:
# Пример: посмотреть похожие слова (для отладки)
# Берём первые 10 слов из словаря модели
#sample_words = model_w2v.wv.index_to_key[:10]
sample_words = model_w2v.wv.index2word[:10]
if sample_words:
    w0 = sample_words[0]
    print(f"Пример похожих слов для '{w0}':")
    try:
        print(model_w2v.wv.most_similar(w0, topn=5))
    except KeyError:
        pass

Пример похожих слов для 'это':
[('уродуют', 0.48425477743148804), ('выглядите', 0.47556495666503906), ('инстинкты', 0.45710456371307373), ('комплимент', 0.45409488677978516), ('оползень', 0.45133155584335327)]


In [32]:
target = "губернатор"

# 1) Проверка, есть ли слово в словаре
if target not in model_w2v.wv:
    print(f"Слово '{target}' отсутствует в словаре модели.")
else:
    # 2) Поиск наиболее похожих слов
    similar = model_w2v.wv.most_similar(positive=[target], topn=10)
    print(f"Похожие слова на '{target}':")
    for word, score in similar:
        print(word, score)

Похожие слова на 'губернатор':
губер 0.5919504165649414
болтун 0.5732956528663635
солнцеликий 0.5558738708496094
советуется 0.5277374982833862
шта 0.5271037220954895
частник 0.5261081457138062
компетентный 0.5251644849777222
кормчий 0.5229372978210449
шоумен 0.5180802941322327
забалуешь 0.5180643796920776


In [24]:
# ---------------------------
# 5. Векторизация документов
# ---------------------------

print("Векторизация документов (усреднение векторов слов)...")

def doc_vector(tokens, model):
    vecs = [model.wv[t] for t in tokens if t in model.wv]
    if len(vecs) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vecs, axis=0)

X = np.vstack([doc_vector(tokens, model_w2v) for tokens in docs_tokens])
y = np.array(labels)

# Преобразуем метки в 0..K-1, если они строковые
unique_labels = np.unique(y)
label2idx = {lab: i for i, lab in enumerate(unique_labels)}
idx2label = {i: lab for lab, i in label2idx.items()}
y_idx = np.array([label2idx[lab] for lab in y])

print(f"Форма X: {X.shape}, количество классов: {len(unique_labels)}")

Векторизация документов (усреднение векторов слов)...
Форма X: (44879, 200), количество классов: 2


In [25]:
# ---------------------------
# 6. Разбиение на train/test
# ---------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y_idx, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_idx
)

print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")

# ---------------------------
# 7. Подготовка данных для PyTorch
# ---------------------------

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

train_ds = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)


Train: 31415, Test: 13464


In [26]:
# ---------------------------
# 8.  ИНС
# ---------------------------

class TextClassifierNN(nn.Module):
    def __init__(self, input_dim: int, num_classes: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, HIDDEN_DIM),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_DIM, num_classes),
        )

    def forward(self, x):
        return self.net(x)

input_dim = X_train.shape[1]
num_classes = len(unique_labels)

model = TextClassifierNN(input_dim=input_dim, num_classes=num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [27]:
# ---------------------------
# 9. Обучение модели
# ---------------------------

print("Обучение нейросети...")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

for epoch in range(EPOCHS_NN):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)
        preds = torch.argmax(logits, dim=1)
        correct += (preds == yb).sum().item()
        total += yb.size(0)

    epoch_loss = total_loss / total
    epoch_acc = correct / total
    print(f"Epoch {epoch+1:02d}/{EPOCHS_NN} | Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.4f}")

Обучение нейросети...
Epoch 01/10 | Loss: 0.3038 | Train Acc: 0.8742
Epoch 02/10 | Loss: 0.1854 | Train Acc: 0.9310
Epoch 03/10 | Loss: 0.1532 | Train Acc: 0.9436
Epoch 04/10 | Loss: 0.1356 | Train Acc: 0.9506
Epoch 05/10 | Loss: 0.1250 | Train Acc: 0.9536
Epoch 06/10 | Loss: 0.1162 | Train Acc: 0.9579
Epoch 07/10 | Loss: 0.1077 | Train Acc: 0.9596
Epoch 08/10 | Loss: 0.1020 | Train Acc: 0.9624
Epoch 09/10 | Loss: 0.0975 | Train Acc: 0.9651
Epoch 10/10 | Loss: 0.0919 | Train Acc: 0.9669


In [29]:
# ---------------------------
# 10. Оценка на тесте
# ---------------------------

print("Оценка на тестовой выборке...")

model.eval()
with torch.no_grad():
    X_test_dev = X_test_t.to(device)
    y_test_dev = y_test_t.to(device)

    logits = model(X_test_dev)
    preds = torch.argmax(logits, dim=1).cpu().numpy()
    y_true = y_test_t.cpu().numpy()

acc = accuracy_score(y_true, preds)
print(f"Test Accuracy: {acc:.4f}")

print("\nClassification report (по индексам классов):")
print(classification_report(y_true, preds, digits=4))

cm = confusion_matrix(y_true, preds)
print("\nConfusion matrix (по индексам классов):")
print(cm)

# Если хотите видеть названия классов вместо индексов:
target_names = [str(idx2label[i]) for i in range(num_classes)]

print("\nClassification report (по названиям классов):")
print(classification_report(y_true, preds, target_names=target_names, digits=4))

Оценка на тестовой выборке...
Test Accuracy: 0.9534

Classification report (по индексам классов):
              precision    recall  f1-score   support

           0     0.9592    0.8319    0.8911      3082
           1     0.9520    0.9895    0.9704     10382

    accuracy                         0.9534     13464
   macro avg     0.9556    0.9107    0.9307     13464
weighted avg     0.9537    0.9534    0.9522     13464


Confusion matrix (по индексам классов):
[[ 2564   518]
 [  109 10273]]

Classification report (по названиям классов):
              precision    recall  f1-score   support

           0     0.9592    0.8319    0.8911      3082
           1     0.9520    0.9895    0.9704     10382

    accuracy                         0.9534     13464
   macro avg     0.9556    0.9107    0.9307     13464
weighted avg     0.9537    0.9534    0.9522     13464



In [31]:
# ---------------------------
# 11. Сохранение модели (опционально)
# ---------------------------

# Сохранить word2vec
model_w2v.save("word2vec_model.model")
print("Модель word2vec сохранена в word2vec_model.model")

# Сохранить веса ИНС
torch.save(model.state_dict(), "nn_classifier.pth")
print("Веса нейросети сохранены в nn_classifier.pth")


Модель word2vec сохранена в word2vec_model.model
Веса нейросети сохранены в nn_classifier.pth
